# LiteTorch Stage 4a Phase 2 — Real TPU Multi-Chip Proof (single submission, Save & Run All)

**What this proves (if it passes):** a standalone C++ program that talks to `libtpu` **directly** through the PJRT C API (no torch_xla, no JAX in the multichip path) compiles three StableHLO programs with explicit sharding annotations, executes them sharded across all 8 TPU chips via PJRT multi-device execution, and matches the PyTorch/XLA references.

**Programs under test:** all-reduce (manual `stablehlo.all_reduce`, channel handle 1), row-shard matmul (communication-free), K-shard matmul (partitioner-inserted combining collective). One 8-device mesh, `num_partitions=8`, `replica_count=1`.

**Roles — read carefully:**
- **Multichip under test:** `multichip_matmul` (PJRT C API + libtpu only).
- **Reference standard (numerics GATE):** PyTorch/XLA. All-reduce gate: multichip vs PyTorch/XLA SPMD all-reduce on identical 1024x1024 FP32 inputs per device, max abs diff < 1e-4. Sharded-matmul gates: gathered 8-device result vs unsharded PyTorch/XLA on identical 512x512 FP32 inputs, max abs diff < 1e-4.
- **JAX:** perf-ceiling reference (what a mature XLA stack tops out at on this host) and a *secondary* numerics cross-check only. JAX is NOT the gate. JAX multi-device all-reduce (`pmap` + `lax.psum`) is the ceiling for the bandwidth table.
- **Host CPU (numpy):** the CPU perf signature and the exact CPU-sum cross-check for the all-reduce gate.
- **Phase-1 spike reference:** single-device 4096^3 FP32 median 1.098 ms (labeled historical reference from the spike v2 PASS run).

**Process isolation:** libtpu/PJRT init is process-global. The multichip binary is its own process; torch_xla and JAX each run as separate subprocesses writing partial JSONs. A baseline failure is recorded and never takes the multichip results down.

**Phase 2 is multi-chip only.** No LiteTorch integration work (that is the real-backend build, after this proof passes).


## Cell 1 — Probe the Kaggle TPU image (libtpu, PJRT API version, packages, host)


In [ ]:
import ctypes, glob, importlib.metadata, json, os, platform, subprocess, sys

probe = {}
probe["python"] = sys.version
probe["platform"] = platform.platform()
probe["nproc"] = os.cpu_count()
try:
    with open("/proc/meminfo") as f:
        probe["meminfo_head"] = [next(f).strip() for _ in range(3)]
except Exception as e:
    probe["meminfo_error"] = str(e)
for pkg in ["torch", "torch_xla", "jax", "jaxlib", "libtpu", "numpy"]:
    try:
        probe["pkg_" + pkg] = importlib.metadata.version(pkg)
    except Exception:
        probe["pkg_" + pkg] = None
probe["env_PJRT_DEVICE"] = os.environ.get("PJRT_DEVICE")
probe["env_TPU_LIBRARY_PATH"] = os.environ.get("TPU_LIBRARY_PATH")
cands = []
for pat in ["/lib/libtpu.so", "/usr/lib/libtpu.so", "/usr/local/lib/libtpu.so",
            "/usr/local/lib/python3.*/dist-packages/libtpu/libtpu.so*",
            "/usr/local/lib/python3.*/site-packages/libtpu/libtpu.so*",
            os.path.expanduser("~/.local/lib/python3.*/site-packages/libtpu/libtpu.so*")]:
    cands.extend(glob.glob(pat))
probe["libtpu_candidates"] = [{"path": p, "size": os.path.getsize(p)} for p in sorted(set(cands))]
probe["pjrt_api_version_ctypes"] = None
if cands:
    try:
        lib = ctypes.CDLL(sorted(set(cands))[0])
        lib.GetPjrtApi.restype = ctypes.c_void_p
        api_ptr = lib.GetPjrtApi()
        major = ctypes.c_int.from_address(api_ptr + 32).value
        minor = ctypes.c_int.from_address(api_ptr + 36).value
        probe["pjrt_api_version_ctypes"] = str(major) + "." + str(minor)
    except Exception as e:
        probe["pjrt_api_version_ctypes_error"] = str(e)
print(json.dumps(probe, indent=2))
with open("/kaggle/working/probe.json", "w") as f:
    json.dump(probe, f, indent=2)

## Cell 2 — Fetch run inputs from kaggle-run-inputs (GitHub-fetch only), sha256-verify every file, build the multichip binary


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, urllib.request

MANIFEST = {
  "multichip_matmul.cpp": "3a5d2352ec9a4ebda581c34e056ecb9763596675363be0ae21fdfb6a24f74b98",
  "pjrt_c_api.h": "8c54c8e030ea88fcba627dd83286e09edbc65aaa34f7aced36da51a48fe83328",
  "run_multichip.py": "0c905224dadc10598e38536ac40b6fbfe5781eef8e1fa58199e1419df54a6c5d",
  "bench_torch_xla.py": "90c874f8182c77e91b47305df3f221ece294736c5d0c4667ec79cb445d3d468b",
  "bench_jax.py": "67d74f6c8ea9af29bfbf9951ff1ae1a87fd66048df4d25e779b224a66503fee5",
  "bench_cpu.py": "ce2dd63304585e7399517f67621c3f7b7bbbf11ceddc252b7f4ea25b3d69e368"
}
REPO = "https://github.com/nguyenminh20000/Litetorch-"
RAW = "https://raw.githubusercontent.com/nguyenminh20000/Litetorch-/kaggle-run-inputs/tpu-multichip/"
SRC = "/kaggle/working/tpu-multichip-src"
os.makedirs(SRC, exist_ok=True)

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

fetched = {}
clone_dir = "/kaggle/working/litetorch-fetch"
clone_cmd = ["git", "clone", "--depth", "1", "--branch", "kaggle-run-inputs", REPO, clone_dir]
print("fetch command:", " ".join(clone_cmd))
if not os.path.exists(os.path.join(clone_dir, "tpu-multichip")):
    r = subprocess.run(clone_cmd, capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-500:])
for name in MANIFEST:
    dst = os.path.join(SRC, name)
    src_clone = os.path.join(clone_dir, "tpu-multichip", name)
    if os.path.exists(src_clone):
        shutil.copyfile(src_clone, dst)
    else:
        urllib.request.urlretrieve(RAW + name, dst)
    got = sha256_of(dst)
    assert got == MANIFEST[name], "SHA256 MISMATCH for " + name + ": " + got
    fetched[name] = got
    print("verified", name, got[:12])
build_cmd = ["g++", "-std=c++14", "-O2", "-o", "/kaggle/working/multichip_matmul",
             os.path.join(SRC, "multichip_matmul.cpp"), "-ldl"]
print("build command:", " ".join(build_cmd))
build = subprocess.run(build_cmd, capture_output=True, text=True)
print(build.stdout, build.stderr)
assert build.returncode == 0, "multichip build failed"
print("BUILD OK -> /kaggle/working/multichip_matmul")


## Cell 3 — Run the multichip binary (own process): all-reduce gate + bandwidth sweep, row-shard and K-shard matmul gates, scaling legs, 1-device controls


In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "/kaggle/working/tpu-multichip-src/run_multichip.py",
                    "--bin", "/kaggle/working/multichip_matmul", "--workdir", "/kaggle/working"],
                   capture_output=True, text=True)
print(r.stdout[-8000:])
print(r.stderr[-3000:])
print("run_multichip returncode:", r.returncode)


## Cell 4 — PyTorch/XLA baselines (separate subprocesses): SPMD all-reduce reference in its own fresh process, then unsharded matmul references on the shared inputs in a second fresh process


In [ ]:
import os, subprocess, sys
os.environ["LIBTPU_INIT_ARGS"] = "--xla_tpu_reserve_recv_ready_sync_flags=true"
r1 = subprocess.run([sys.executable, "/kaggle/working/tpu-multichip-src/bench_torch_xla.py",
                     "--workdir", "/kaggle/working", "--spmd-allreduce-only"], capture_output=True, text=True)
print(r1.stdout[-6000:])
print(r1.stderr[-3000:])
print("bench_torch_xla --spmd-allreduce-only returncode:", r1.returncode)
r = subprocess.run([sys.executable, "/kaggle/working/tpu-multichip-src/bench_torch_xla.py",
                    "--workdir", "/kaggle/working"], capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-3000:])
print("bench_torch_xla returncode:", r.returncode)


## Cell 5 — JAX baselines (separate subprocess): perf ceiling + secondary cross-check only; pmap all-reduce and pmap row-shard matmul


In [ ]:
import os, subprocess, sys
os.environ["LIBTPU_INIT_ARGS"] = "--xla_tpu_reserve_recv_ready_sync_flags=true"
r = subprocess.run([sys.executable, "/kaggle/working/tpu-multichip-src/bench_jax.py",
                    "--workdir", "/kaggle/working"], capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-3000:])
print("bench_jax returncode:", r.returncode)


## Cell 6 — Host CPU signature (separate subprocess, numpy FP32)


In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "/kaggle/working/tpu-multichip-src/bench_cpu.py",
                    "--workdir", "/kaggle/working"], capture_output=True, text=True)
print(r.stdout[-6000:])
print(r.stderr[-3000:])
print("bench_cpu returncode:", r.returncode)


## Cell 7 — Merge partials, evaluate the Phase 2 proof criteria (DESIGN.md §11.7), write results.json


In [ ]:
import json, os
import numpy as np

W = "/kaggle/working"
def load(name):
    p = os.path.join(W, name)
    if os.path.exists(p):
        with open(p) as f:
            return json.load(f)
    return {"error": "partial file missing: " + name}

results = {}
results["stage"] = "4a-phase2-multichip"
results["roles"] = {
    "multichip_under_test": "multichip_matmul via PJRT C API against libtpu directly; no torch_xla, no JAX in this path",
    "numerics_gate": "PyTorch/XLA is the reference standard; all-reduce and sharded-matmul gates are multichip vs PyTorch/XLA on identical inputs, max abs diff < 1e-4 FP32",
    "jax": "perf ceiling reference and secondary numerics cross-check only; NOT the gate",
    "host_cpu": "numpy FP32 signature for the perf-inconsistency criterion and the exact CPU-sum cross-check",
    "spike_reference": {"label": "Phase-1 spike v2 PASS run, historical reference, not re-measured in this run",
                        "f32_4096_median_ms": 1.098},
}
results["probe"] = load("probe.json")
results["multichip"] = load("partial_multichip.json")
results["torch_xla"] = load("partial_torch_xla.json")
results["jax"] = load("partial_jax.json")
results["cpu"] = load("partial_cpu.json")

hard = []
mc = results["multichip"]
mc_runs = mc.get("runs", {}) if isinstance(mc, dict) else {}

def matmul_gate(mc_file, tx_file):
    if os.path.exists(mc_file) and os.path.exists(tx_file):
        s = np.fromfile(mc_file, dtype=np.float32)
        t = np.fromfile(tx_file, dtype=np.float32)
        if s.size == 512 * 512 and s.shape == t.shape:
            d = float(np.abs(s - t).max())
            return {"max_abs_diff": d, "tolerance": 1e-4, "status": "PASS" if d < 1e-4 else "FAIL"}
    return {"status": "not_evaluated"}

ar = {"status": "not_evaluated"}
mc_ar = os.path.join(W, "out_allreduce_1024.bin")
tx_ar = os.path.join(W, "out_torch_xla_allreduce_1024.bin")
ar_in = os.path.join(W, "allreduce_inputs_1024.bin")
if os.path.exists(mc_ar) and os.path.exists(tx_ar):
    s = np.fromfile(mc_ar, dtype=np.float32)
    t = np.fromfile(tx_ar, dtype=np.float32)
    if s.size == 8 * 1024 * 1024 and s.shape == t.shape:
        per_dev = [float(np.abs(s[d*1024*1024:(d+1)*1024*1024] - t[d*1024*1024:(d+1)*1024*1024]).max()) for d in range(8)]
        d = max(per_dev)
        ar = {"allreduce_fp32_1024_max_abs_diff": d, "per_device_max_abs_diff": per_dev,
              "tolerance": 1e-4, "allreduce_fp32_1024_status": "PASS" if d < 1e-4 else "FAIL"}
        if os.path.exists(ar_in):
            ins = np.fromfile(ar_in, dtype=np.float32).reshape(8, 1024, 1024)
            cpu_sum = ins.sum(axis=0).reshape(-1).astype(np.float32)
            cd = float(np.abs(s - np.tile(cpu_sum, 8)).max())
            ar["allreduce_fp32_1024_vs_cpu_sum_max_abs_diff"] = cd
            ar["vs_cpu_sum_status"] = "PASS" if cd < 1e-4 else "FAIL"
results["allreduce_fp32_1024"] = ar
hard.append(ar.get("allreduce_fp32_1024_status", "not_evaluated"))
if "vs_cpu_sum_status" in ar:
    hard.append(ar["vs_cpu_sum_status"])

bw = {}
for size in (512, 2048, 4096):
    key = "allreduce_bw_%dmb" % (size * size * 4 // (1024 * 1024))
    run_key = {512: "allreduce_bw_1mb", 2048: "allreduce_bw_16mb", 4096: "allreduce_bw_64mb"}[size]
    entry = {"status": "not_evaluated"}
    try:
        med = mc_runs[run_key]["result"]["execute_median_ms"]
        nbytes = size * size * 4
        entry = {"bytes_per_device": nbytes, "median_ms": med,
                 "gbps": nbytes / med / 1e6, "latency_ms": med, "status": "reported"}
    except Exception as e:
        entry = {"status": "not_evaluated", "reason": str(e)}
    bw[key] = entry
results["allreduce_bandwidth"] = bw
bw_jax = {}
try:
    for size in (512, 2048, 4096):
        j = results["jax"]["allreduce_pmap"]["bw_%d" % size]
        bw_jax["allreduce_bw_%dmb" % (size * size * 4 // (1024 * 1024))] = {
            "bytes_per_device": j["bytes_per_device"], "median_ms": j["execute_median_ms"],
            "gbps": j["gbps"], "latency_ms": j["execute_median_ms"], "status": "reported"}
except Exception as e:
    bw_jax = {"status": "not_evaluated", "reason": str(e)}
results["allreduce_bandwidth_jax"] = bw_jax

row_gate = matmul_gate(os.path.join(W, "out_rowshard_512.bin"), os.path.join(W, "out_torch_xla_512.bin"))
row = {"matmul_rowshard_fp32_512_status": row_gate["status"]}
if "max_abs_diff" in row_gate:
    row["matmul_rowshard_fp32_512_max_abs_diff"] = row_gate["max_abs_diff"]
    row["tolerance"] = 1e-4
results["matmul_rowshard_fp32_512"] = row
hard.append(row["matmul_rowshard_fp32_512_status"])

def scaling(key8, key1):
    try:
        m8 = mc_runs[key8]["result"]["execute_median_ms"]
        m1 = mc_runs[key1]["result"]["execute_median_ms"]
        ratio = m8 / m1
        return {"ratio": ratio, "median_8dev_ms": m8, "median_1dev_ms": m1,
                "status": "PASS" if ratio < 0.5 else "FAIL"}
    except Exception as e:
        return {"status": "not_evaluated", "reason": str(e)}

sc_bf16 = scaling("matmul_rowshard_bf16_8192", "matmul_rowshard_bf16_8192_1dev")
results["matmul_rowshard_bf16_8192"] = {
    "matmul_rowshard_bf16_8192_ratio": sc_bf16.get("ratio"),
    "matmul_rowshard_bf16_8192_status": sc_bf16["status"],
    "median_8dev_ms": sc_bf16.get("median_8dev_ms"),
    "median_1dev_ms": sc_bf16.get("median_1dev_ms")}
hard.append(sc_bf16["status"])
sc_f32 = scaling("matmul_rowshard_fp32_8192", "matmul_rowshard_fp32_8192_1dev")
results["matmul_rowshard_fp32_8192"] = {
    "matmul_rowshard_fp32_8192_ratio": sc_f32.get("ratio"),
    "matmul_rowshard_fp32_8192_status": sc_f32["status"],
    "median_8dev_ms": sc_f32.get("median_8dev_ms"),
    "median_1dev_ms": sc_f32.get("median_1dev_ms")}
hard.append(sc_f32["status"])

k_gate = matmul_gate(os.path.join(W, "out_kshard_512.bin"), os.path.join(W, "out_torch_xla_512.bin"))
krow = {"matmul_kshard_fp32_512_status": k_gate["status"]}
if "max_abs_diff" in k_gate:
    krow["matmul_kshard_fp32_512_max_abs_diff"] = k_gate["max_abs_diff"]
    krow["tolerance"] = 1e-4
results["matmul_kshard_fp32_512"] = krow
hard.append(krow["matmul_kshard_fp32_512_status"])

k_bf16 = {"status": "not_evaluated"}
mc_bf = os.path.join(W, "out_kshard_bf16_4096.bin")
tx_bf = os.path.join(W, "out_torch_xla_bf16_4096.bin")
if os.path.exists(mc_bf) and os.path.exists(tx_bf):
    s = np.fromfile(mc_bf, dtype=np.float32)
    t = np.fromfile(tx_bf, dtype=np.float32)
    if s.size == 4096 * 4096 and s.shape == t.shape:
        r = float((np.abs(s - t) / np.maximum(1.0, np.abs(t))).max())
        k_bf16 = {"matmul_kshard_bf16_4096_max_rel_err": r, "tolerance": 1e-2,
                  "matmul_kshard_bf16_4096_status": "PASS" if r < 1e-2 else "FAIL"}
results["matmul_kshard_bf16_4096"] = k_bf16
hard.append(k_bf16.get("matmul_kshard_bf16_4096_status", "not_evaluated"))

pdb = {}
for run_key, label, shard_map in (
        ("allreduce_gate_1024", "allreduce", {"a": "replicated", "c": "replicated"}),
        ("matmul_rowshard_fp32_8192", "matmul_rowshard", {"a": "sharded", "b": "replicated", "c": "sharded"}),
        ("matmul_kshard_fp32_4096", "matmul_kshard", {"a": "sharded", "b": "sharded", "c": "replicated"})):
    entry = {"status": "not_evaluated"}
    try:
        res = mc_runs[run_key]["result"]
        gb = res["global_bytes"]
        odb = res["on_device_bytes"]
        checks = {}
        ok = True
        for role, kind in shard_map.items():
            per = odb[role]
            exp = gb[role] // 8 if kind == "sharded" else gb[role]
            good = all(v == exp for v in per) and len(per) == 8
            ok = ok and good
            checks[role] = {"global_bytes": gb[role], "per_device_bytes": per,
                            "expected": exp, "status": "PASS" if good else "FAIL"}
        entry = {"checks": checks, "status": "PASS" if ok else "FAIL"}
    except Exception as e:
        entry = {"status": "not_evaluated", "reason": str(e)}
    pdb[label] = entry
    hard.append(entry["status"])
results["per_device_bytes"] = pdb

place = {"status": "not_evaluated"}
try:
    info = mc_runs["allreduce_gate_1024"]["info_lines"]
    kinds = []
    ndev = None
    for line in info:
        if line.startswith("num_addressable_devices="):
            ndev = int(line.split("=")[1])
        if line.startswith("device["):
            kinds.append(line.split("kind=")[1])
    res = mc_runs["allreduce_gate_1024"]["result"]
    nout = res["num_outputs_per_device"] * res["num_devices"]
    ok = (ndev == 8 and len(kinds) == 8 and all(k == "TPU v5 lite" for k in kinds)
          and res["num_devices"] == 8 and nout == 8)
    place = {"num_addressable_devices": ndev, "device_kinds": kinds,
             "per_device_output_buffers": nout,
             "status": "PASS" if ok else "FAIL"}
except Exception as e:
    place = {"status": "not_evaluated", "reason": str(e)}
results["placement"] = place
hard.append(place["status"])

compile_ms = {}
for run_key, label in (("allreduce_gate_1024", "allreduce"),
                       ("matmul_rowshard_fp32_8192", "matmul_rowshard"),
                       ("matmul_kshard_fp32_4096", "matmul_kshard")):
    try:
        compile_ms[label] = mc_runs[run_key]["result"]["compile_ms"]
    except Exception:
        compile_ms[label] = None
results["compile_ms"] = compile_ms
results["compile_ms_over_30s_flag"] = any(v is not None and v > 30000 for v in compile_ms.values())

sanity = {}
for key in ("torch_xla", "jax", "cpu"):
    entry = {"status": "not_evaluated"}
    try:
        runs = results[key]["runs"]
        m512 = runs["f32_512"]["execute_median_ms"]
        m4096 = runs["f32_4096"]["execute_median_ms"]
        entry = {"f32_512_median_ms": m512, "f32_4096_median_ms": m4096,
                 "ratio_4096_over_512": m4096 / m512,
                 "flag": "SUSPECT_UNSYNCHRONIZED" if m4096 < 2.0 * m512 else "ok",
                 "note": "informational only; a synchronized 4096^3 step does 512x the FLOPs of 512^3, so a ratio near 1 indicates unsynchronized timing"}
    except Exception as e:
        entry = {"status": "not_evaluated", "reason": str(e)}
    sanity[key] = entry
results["timing_sanity"] = sanity

spike_vs_cpu = {"status": "not_evaluated"}
mc_out = os.path.join(W, "out_rowshard_512.bin")
gate_a = os.path.join(W, "gate_a_512.bin")
gate_b = os.path.join(W, "gate_b_512.bin")
if os.path.exists(mc_out) and os.path.exists(gate_a) and os.path.exists(gate_b):
    s = np.fromfile(mc_out, dtype=np.float32)
    if s.size == 512 * 512:
        a_ref = np.fromfile(gate_a, dtype=np.float32).reshape(512, 512)
        b_ref = np.fromfile(gate_b, dtype=np.float32).reshape(512, 512)
        ref = (a_ref @ b_ref).reshape(-1)
        spike_vs_cpu = {"max_abs_diff": float(np.abs(s - ref).max()),
                        "note": "informational only; multichip row-shard output vs host-CPU FP32 reference on the shared gate inputs; NOT the gate (P5 tracking)"}
results["spike_vs_torch_cpu_fp32_max_abs_diff"] = spike_vs_cpu

if any(s == "FAIL" for s in hard):
    verdict = "FAIL"
elif any(s == "not_evaluated" for s in hard):
    verdict = "not_evaluated"
else:
    verdict = "PASS"
results["verdict"] = verdict

with open(os.path.join(W, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
print(json.dumps({"verdict": verdict, "hard_statuses": hard,
                  "allreduce_fp32_1024": ar, "allreduce_bandwidth": bw,
                  "matmul_rowshard_fp32_512": row, "matmul_rowshard_bf16_8192": sc_bf16,
                  "matmul_rowshard_fp32_8192": sc_f32, "matmul_kshard_fp32_512": krow,
                  "matmul_kshard_bf16_4096": k_bf16, "per_device_bytes": pdb,
                  "placement": place, "compile_ms": compile_ms,
                  "timing_sanity": sanity, "spike_vs_torch_cpu_fp32": spike_vs_cpu}, indent=2))
print("results.json written to /kaggle/working/results.json")
